In [1]:
#uid variance entropy
import json
from datasets import Dataset
import os

dataset_path="/scratch/mjgwak/uid-reasoning/scripts/outputs/runs.baselines/aime.deepseek-r1-distill-qwen-1.5b.direct/train.9.3,18:48-8.json"

with open(dataset_path, 'r') as f:
    data = json.load(f)

uid_metrics = [
    "uid_variance_entropy"
]

# Store filtered data for SFT training
sft_data_highest = []
sft_data_lowest = []

for problem in data:
    problem_id = problem.get('id', 'unknown')
    correct_answer = problem.get('answer', '')
    
    # Find all outputs for this problem
    outputs = []
    i = 0
    while f'Output_{i}' in problem:
        output_key = f'Output_{i}'
        metrics_key = f'Metrics_{i}'
        uid_metrics_key = f'uid_metrics_{i}'
        pred_answer_key = f'Pred_Answer_{i}'
        
        if uid_metrics_key in problem:
            # Check if answer is correct
            pred_answer = problem.get(pred_answer_key, '')
            is_correct = pred_answer == correct_answer
            
            outputs.append({
                'index': i,
                'uid_metrics': problem[uid_metrics_key],
                'accuracy': problem[metrics_key].get('acc', 0),
                'exact_match': problem[metrics_key].get('em', 0),
                'f1': problem[metrics_key].get('f1', 0),
                'pred_answer': pred_answer,
                'correct_answer': correct_answer,
                'is_correct': is_correct,
                'output_text': problem[output_key]
            })
        i += 1
    
    if not outputs:
        continue
    
    # Check if any output is correct
    correct_outputs = [out for out in outputs if out['is_correct']]
    
    if correct_outputs:
        # Among correct outputs, find the one with highest UID score for each metric
        best_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with highest UID score among correct answers
                highest_output = max(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, -float('inf'))
                )
                best_outputs[metric] = highest_output
                
        # Create SFT training entry with just the output text
        for metric, output in best_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_highest.append(sft_entry)
        
        worst_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with lowest UID score among correct answers
                lowest_output = min(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, float('inf'))
                )
                worst_outputs[metric] = lowest_output
        
        # Create SFT training entry with just the output text
        for metric, output in worst_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_lowest.append(sft_entry)
        
print(f"Original problems: {len(data)}")
print(f"Highest UID output texts: {len(sft_data_highest)}")
print(f"Lowest UID output texts: {len(sft_data_lowest)}")

# Save filtered data
os.makedirs("sft_data", exist_ok=True)
with open("sft_data/sft_data_variance_highest.json", "w") as f:
    json.dump(sft_data_highest, f, indent=4)
    
with open("sft_data/sft_data_variance_lowest.json", "w") as f:
    json.dump(sft_data_lowest, f, indent=4)

sft_dataset_highest = Dataset.from_list(sft_data_highest)
sft_dataset_lowest = Dataset.from_list(sft_data_lowest)

sft_dataset_highest.push_to_hub("talzoomanzoo/entropy_highest_uid_variance_sft_ds-1.5b")
sft_dataset_lowest.push_to_hub("talzoomanzoo/entropy_lowest_uid_variance_sft_ds-1.5b")

/home/mjgwak/miniconda3/envs/uid_2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original problems: 933
Highest UID output texts: 658
Lowest UID output texts: 658


Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.71s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/entropy_lowest_uid_variance_sft_ds-1.5b/commit/e4eabdd14e040f8b2f08001d04ab1add687afd1f', commit_message='Upload dataset', commit_description='', oid='e4eabdd14e040f8b2f08001d04ab1add687afd1f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/entropy_lowest_uid_variance_sft_ds-1.5b', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/entropy_lowest_uid_variance_sft_ds-1.5b'), pr_revision=None, pr_num=None)

In [2]:
#uid gini entropy
import json
from datasets import Dataset
import os

dataset_path="/scratch/mjgwak/uid-reasoning/scripts/outputs/runs.baselines/aime.deepseek-r1-distill-qwen-1.5b.direct/train.9.3,18:48-8.json"

with open(dataset_path, 'r') as f:
    data = json.load(f)

uid_metrics = [
    "uid_gini_entropy"
]

# Store filtered data for SFT training
sft_data_highest = []
sft_data_lowest = []

for problem in data:
    problem_id = problem.get('id', 'unknown')
    correct_answer = problem.get('answer', '')
    
    # Find all outputs for this problem
    outputs = []
    i = 0
    while f'Output_{i}' in problem:
        output_key = f'Output_{i}'
        metrics_key = f'Metrics_{i}'
        uid_metrics_key = f'uid_metrics_{i}'
        pred_answer_key = f'Pred_Answer_{i}'
        
        if uid_metrics_key in problem and metrics_key in problem:
            # Check if answer is correct
            pred_answer = problem.get(pred_answer_key, '')
            is_correct = pred_answer == correct_answer
            
            outputs.append({
                'index': i,
                'uid_metrics': problem[uid_metrics_key],
                'accuracy': problem[metrics_key].get('acc', 0),
                'exact_match': problem[metrics_key].get('em', 0),
                'f1': problem[metrics_key].get('f1', 0),
                'pred_answer': pred_answer,
                'correct_answer': correct_answer,
                'is_correct': is_correct,
                'output_text': problem[output_key]
            })
        i += 1
    
    if not outputs:
        continue
    
    # Check if any output is correct
    correct_outputs = [out for out in outputs if out['is_correct']]
    
    if correct_outputs:
        # Among correct outputs, find the one with highest UID score for each metric
        best_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with highest UID score among correct answers
                highest_output = max(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, -float('inf'))
                )
                best_outputs[metric] = highest_output
                
        # Create SFT training entry with just the output text
        for metric, output in best_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_highest.append(sft_entry)
        
        worst_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with lowest UID score among correct answers
                lowest_output = min(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, float('inf'))
                )
                worst_outputs[metric] = lowest_output
        
        # Create SFT training entry with just the output text
        for metric, output in worst_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_lowest.append(sft_entry)
        
print(f"Original problems: {len(data)}")
print(f"Highest UID output texts: {len(sft_data_highest)}")
print(f"Lowest UID output texts: {len(sft_data_lowest)}")

# Save filtered data
os.makedirs("sft_data", exist_ok=True)
with open("sft_data/sft_data_gini_highest.json", "w") as f:
    json.dump(sft_data_highest, f, indent=4)
    
with open("sft_data/sft_data_gini_lowest.json", "w") as f:
    json.dump(sft_data_lowest, f, indent=4)

sft_dataset_highest = Dataset.from_list(sft_data_highest)
sft_dataset_lowest = Dataset.from_list(sft_data_lowest)

sft_dataset_highest.push_to_hub("talzoomanzoo/entropy_highest_uid_gini_sft_ds-1.5b")
sft_dataset_lowest.push_to_hub("talzoomanzoo/entropy_lowest_uid_gini_sft_ds-1.5b")

Original problems: 933
Highest UID output texts: 658
Lowest UID output texts: 658


Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.95s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/entropy_lowest_uid_gini_sft_ds-1.5b/commit/ef212cde785069390b5f3afcdaab9e2abaa75896', commit_message='Upload dataset', commit_description='', oid='ef212cde785069390b5f3afcdaab9e2abaa75896', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/entropy_lowest_uid_gini_sft_ds-1.5b', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/entropy_lowest_uid_gini_sft_ds-1.5b'), pr_revision=None, pr_num=None)

In [3]:
#uid shannon entropy
import json
from datasets import Dataset
import os

dataset_path="/scratch/mjgwak/uid-reasoning/scripts/outputs/runs.baselines/aime.deepseek-r1-distill-qwen-1.5b.direct/train.9.3,18:48-8.json"

with open(dataset_path, 'r') as f:
    data = json.load(f)

uid_metrics = [
    "uid_shannon_entropy"
]

# Store filtered data for SFT training
sft_data_highest = []
sft_data_lowest = []

for problem in data:
    problem_id = problem.get('id', 'unknown')
    correct_answer = problem.get('answer', '')
    
    # Find all outputs for this problem
    outputs = []
    i = 0
    while f'Output_{i}' in problem:
        output_key = f'Output_{i}'
        metrics_key = f'Metrics_{i}'
        uid_metrics_key = f'uid_metrics_{i}'
        pred_answer_key = f'Pred_Answer_{i}'
        
        if uid_metrics_key in problem and metrics_key in problem:
            # Check if answer is correct
            pred_answer = problem.get(pred_answer_key, '')
            is_correct = pred_answer == correct_answer
            
            outputs.append({
                'index': i,
                'uid_metrics': problem[uid_metrics_key],
                'accuracy': problem[metrics_key].get('acc', 0),
                'exact_match': problem[metrics_key].get('em', 0),
                'f1': problem[metrics_key].get('f1', 0),
                'pred_answer': pred_answer,
                'correct_answer': correct_answer,
                'is_correct': is_correct,
                'output_text': problem[output_key]
            })
        i += 1
    
    if not outputs:
        continue
    
    # Check if any output is correct
    correct_outputs = [out for out in outputs if out['is_correct']]
    
    if correct_outputs:
        # Among correct outputs, find the one with highest UID score for each metric
        best_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with highest UID score among correct answers
                highest_output = max(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, -float('inf'))
                )
                best_outputs[metric] = highest_output
                
        # Create SFT training entry with just the output text
        for metric, output in best_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_highest.append(sft_entry)
        
        worst_outputs = {}
        
        for metric in uid_metrics:
            # Filter to only correct outputs that have this metric
            correct_outputs_with_metric = [
                out for out in correct_outputs 
                if metric in out['uid_metrics']
            ]
            
            if correct_outputs_with_metric:
                # Find output with lowest UID score among correct answers
                lowest_output = min(
                    correct_outputs_with_metric, 
                    key=lambda x: x['uid_metrics'].get(metric, float('inf'))
                )
                worst_outputs[metric] = lowest_output
        
        # Create SFT training entry with just the output text
        for metric, output in worst_outputs.items():
            sft_entry = {
                'id': problem_id,
                'year': problem.get('year'),
                'problem_number': problem.get('problem_number'),
                'Question': problem.get('Question'),
                'answer': correct_answer,
                'text': output['output_text']
            }
            sft_data_lowest.append(sft_entry)
        
print(f"Original problems: {len(data)}")
print(f"Highest UID output texts: {len(sft_data_highest)}")
print(f"Lowest UID output texts: {len(sft_data_lowest)}")

# Save filtered data
os.makedirs("sft_data", exist_ok=True)
with open("sft_data/sft_data_shannon_highest.json", "w") as f:
    json.dump(sft_data_highest, f, indent=4)
    
with open("sft_data/sft_data_shannon_lowest.json", "w") as f:
    json.dump(sft_data_lowest, f, indent=4)

sft_dataset_highest = Dataset.from_list(sft_data_highest)
sft_dataset_lowest = Dataset.from_list(sft_data_lowest)

sft_dataset_highest.push_to_hub("talzoomanzoo/entropy_highest_uid_shannon_sft_ds-1.5b")
sft_dataset_lowest.push_to_hub("talzoomanzoo/entropy_lowest_uid_shannon_sft_ds-1.5b")

Original problems: 933
Highest UID output texts: 658
Lowest UID output texts: 658


Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.92s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/entropy_lowest_uid_shannon_sft_ds-1.5b/commit/71ce7bf13e9fbaff06becfaddce3dcafa22bd474', commit_message='Upload dataset', commit_description='', oid='71ce7bf13e9fbaff06becfaddce3dcafa22bd474', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/entropy_lowest_uid_shannon_sft_ds-1.5b', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/entropy_lowest_uid_shannon_sft_ds-1.5b'), pr_revision=None, pr_num=None)